# Deploy Ultravox v0.5 (Llama 3.2 1B) to a Databricks Serving Endpoint

This notebook walks through the end-to-end process of deploying the **[fixie-ai/ultravox-v0_5-llama-3_2-1b](https://huggingface.co/fixie-ai/ultravox-v0_5-llama-3_2-1b)** multimodal speech-language model from HuggingFace as a Databricks Model Serving endpoint.

### Steps
1. **Configure** — Set catalog, schema, and model names.
2. **Install dependencies** — Install required Python packages.
3. **Download & validate** — Fetch the model and processor from HuggingFace Hub.
4. **Wrap as custom PyFunc** — Define an `mlflow.pyfunc.PythonModel` with custom inference logic.
5. **Log & register** — Log the model to MLflow and register it in Unity Catalog.
6. **Create serving endpoint** — Spin up a GPU-backed Model Serving endpoint.
7. **Query** — Send a test request to the live endpoint.

> **Requirements**: A Databricks workspace with Unity Catalog enabled and permissions to create models and serving endpoints.

In [0]:
# ---------------------------------------------------------------------------
# Configuration — update these values to match your environment
# ---------------------------------------------------------------------------

HF_MODEL_NAME = "fixie-ai/ultravox-v0_5-llama-3_2-1b"  # HuggingFace model identifier

# Unity Catalog coordinates (change to your own catalog / schema)
CATALOG = "<your_catalog>"          # e.g. "main"
SCHEMA  = "<your_schema>"           # e.g. "ml_models"
MODEL_NAME    = "ultravox_v05_llama32_1b"
ENDPOINT_NAME = "ultravox-v05-llama32-1b"

LOCAL_MODEL_PATH = "/tmp/ultravox_model"  # temp directory for downloaded weights

In [0]:
%pip install -U transformers torch accelerate mlflow sentencepiece protobuf
dbutils.library.restartPython()

In [0]:
# ---------------------------------------------------------------------------
# After restartPython() all variables are cleared — re-import and re-define.
# ---------------------------------------------------------------------------
import os, time, json
import pandas as pd
import mlflow
import mlflow.pyfunc
from mlflow.models import infer_signature, ModelSignature
from mlflow.types.schema import Schema, ColSpec

# Re-define constants (must match the Configuration cell above)
HF_MODEL_NAME  = "fixie-ai/ultravox-v0_5-llama-3_2-1b"
CATALOG        = "<your_catalog>"
SCHEMA         = "<your_schema>"
MODEL_NAME     = "ultravox_v05_llama32_1b"
ENDPOINT_NAME  = "ultravox-v05-llama32-1b"
LOCAL_MODEL_PATH = "/tmp/ultravox_model"

REGISTERED_MODEL_NAME = f"{CATALOG}.{SCHEMA}.{MODEL_NAME}"
print(f"Registered model name: {REGISTERED_MODEL_NAME}")

## Step 1 — Download & Validate the Model

Download the model weights and processor from HuggingFace Hub and persist them to a local temp directory. We use `trust_remote_code=True` because Ultravox ships custom modelling code.

In [0]:
from transformers import AutoModel, AutoProcessor, AutoTokenizer
import torch

print(f"Downloading model: {HF_MODEL_NAME} ...")

# Download model weights (half-precision to save memory)
model = AutoModel.from_pretrained(
    HF_MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=torch.float16,
)

# Download processor (handles both text tokenization and audio features)
processor = AutoProcessor.from_pretrained(
    HF_MODEL_NAME,
    trust_remote_code=True,
)

# Save locally so we can package them as MLflow artifacts
model.save_pretrained(LOCAL_MODEL_PATH)
processor.save_pretrained(LOCAL_MODEL_PATH)

print(f"Model and processor saved to {LOCAL_MODEL_PATH}")
print(f"Model type : {type(model).__name__}")
print(f"Processor type: {type(processor).__name__}")
print(f"Model params  : {sum(p.numel() for p in model.parameters()):,}")

## Step 2 — Define a Custom PyFunc Model

Because Ultravox uses custom architecture code and multimodal inputs that don't map to a standard `transformers` pipeline task, we wrap it in an **MLflow PyFunc** model. This gives us full control over:

* How the model and processor are loaded (`load_context`)
* How inputs are preprocessed and inference is executed (`predict`)

> **Note**: Adapt the `predict` method to match your production input format (e.g., audio bytes, URLs, or pre-extracted features).

In [0]:
class UltravoxPyfuncModel(mlflow.pyfunc.PythonModel):
    """
    Custom MLflow PyFunc wrapper for the Ultravox multimodal model.
    Accepts text input and returns generated text.
    """

    def load_context(self, context):
        """Load model and processor once when the serving container starts."""
        from transformers import AutoModel, AutoProcessor
        import torch

        model_path = context.artifacts["model_path"]

        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        self.model = AutoModel.from_pretrained(
            model_path,
            trust_remote_code=True,
            torch_dtype=torch.float16,
        ).to(self.device)
        self.model.eval()

        self.processor = AutoProcessor.from_pretrained(
            model_path,
            trust_remote_code=True,
        )
        print(f"Model loaded on {self.device}")

    def predict(self, context, model_input: pd.DataFrame) -> pd.DataFrame:
        """
        Run inference on a batch of text inputs.

        Parameters
        ----------
        model_input : pd.DataFrame
            Must contain a 'text' column with the input prompts.

        Returns
        -------
        pd.DataFrame with a 'generated_text' column.
        """
        import torch

        texts = model_input["text"].tolist()
        results = []

        for text in texts:
            inputs = self.processor(
                text=text,
                return_tensors="pt",
            ).to(self.device)

            with torch.no_grad():
                output_ids = self.model.generate(
                    **inputs,
                    max_new_tokens=256,
                    do_sample=False,
                )

            generated = self.processor.batch_decode(
                output_ids, skip_special_tokens=True
            )
            results.append(generated[0] if generated else "")

        return pd.DataFrame({"generated_text": results})


print("UltravoxPyfuncModel class defined.")

## Step 3 — Log & Register the Model in Unity Catalog

We log the custom PyFunc model to MLflow with:
* **`artifacts`** pointing to the local model directory
* **`signature`** and **`input_example`** for serving compatibility
* **`pip_requirements`** to pin the serving environment
* **`registered_model_name`** to auto-register in Unity Catalog

In [0]:
mlflow.set_registry_uri("databricks-uc")

# Define the model signature explicitly
input_schema = Schema([ColSpec("string", "text")])
output_schema = Schema([ColSpec("string", "generated_text")])
signature = ModelSignature(inputs=input_schema, outputs=output_schema)

input_example = pd.DataFrame({"text": ["Describe the weather today."]})

with mlflow.start_run(run_name="ultravox-v05-registration") as run:
    model_info = mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=UltravoxPyfuncModel(),
        artifacts={"model_path": LOCAL_MODEL_PATH},
        signature=signature,
        input_example=input_example,
        pip_requirements=[
            "transformers>=4.40.0",
            "torch>=2.0.0",
            "accelerate>=0.25.0",
            "sentencepiece",
            "protobuf",
        ],
        registered_model_name=REGISTERED_MODEL_NAME,
    )
    print(f"MLflow Run ID   : {run.info.run_id}")
    print(f"Model URI       : {model_info.model_uri}")
    print(f"Registered as   : {REGISTERED_MODEL_NAME}")

## Step 4 — Create a GPU-Backed Serving Endpoint

Use the MLflow Deployments SDK to create a Model Serving endpoint pointing to the registered Unity Catalog model. The endpoint uses **GPU workload type** for efficient LLM inference.

In [0]:
import mlflow.deployments

client = mlflow.deployments.get_deploy_client("databricks")

endpoint_config = {
    "served_entities": [
        {
            "entity_name": REGISTERED_MODEL_NAME,
            "entity_version": "1",
            "workload_size": "Small",
            "workload_type": "GPU_SMALL",  # GPU-backed for LLM inference
            "scale_to_zero_enabled": True,
        }
    ],
    "traffic_config": {
        "routes": [
            {
                "served_model_name": f"{MODEL_NAME}-1",
                "traffic_percentage": 100,
            }
        ]
    },
}

try:
    endpoint = client.create_endpoint(
        name=ENDPOINT_NAME,
        config=endpoint_config,
    )
    print(f"Endpoint '{ENDPOINT_NAME}' creation initiated.")
    print(json.dumps(endpoint, indent=2, default=str))
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"Endpoint '{ENDPOINT_NAME}' already exists. Updating config...")
        endpoint = client.update_endpoint(
            endpoint=ENDPOINT_NAME,
            config=endpoint_config,
        )
    else:
        raise e

## Step 5 — Wait for the Endpoint to Become Ready

Serving endpoints take several minutes to provision (downloading model artifacts, building the container, allocating GPU). The cell below polls until the state is `READY`.

In [0]:
POLL_INTERVAL_SECONDS = 60
MAX_WAIT_MINUTES = 45

print(f"Waiting for endpoint '{ENDPOINT_NAME}' to become READY ...")
print(f"(polling every {POLL_INTERVAL_SECONDS}s, max wait {MAX_WAIT_MINUTES} min)\n")

start = time.time()
while True:
    endpoint_status = client.get_endpoint(ENDPOINT_NAME)
    state = endpoint_status.get("state", {}).get("ready", "UNKNOWN")
    config_update = endpoint_status.get("state", {}).get("config_update", "UNKNOWN")
    elapsed = (time.time() - start) / 60

    print(f"  [{elapsed:5.1f} min] ready={state}  config_update={config_update}")

    if state == "READY":
        print(f"\nEndpoint '{ENDPOINT_NAME}' is READY!")
        break
    if elapsed > MAX_WAIT_MINUTES:
        print(f"\nTimed out after {MAX_WAIT_MINUTES} minutes. Check the Serving UI.")
        break

    time.sleep(POLL_INTERVAL_SECONDS)

## Step 6 — Query the Serving Endpoint

Send a test request to verify the endpoint is working.

In [0]:
# Send a sample inference request
response = client.predict(
    endpoint=ENDPOINT_NAME,
    inputs={
        "dataframe_records": [
            {"text": "Explain what a transformer model is in simple terms."},
            {"text": "Summarize the benefits of multimodal AI."},
        ]
    },
)

print("Response from the serving endpoint:")
print(json.dumps(response, indent=2, default=str))

## Cleanup (Optional)

Uncomment and run the cell below to **delete** the serving endpoint when you no longer need it. This stops billing for the endpoint compute.

In [0]:
# -----------------------------------------------------------------------
# CAUTION: Uncomment the lines below to permanently delete the endpoint.
# -----------------------------------------------------------------------

# client.delete_endpoint(ENDPOINT_NAME)
# print(f"Endpoint '{ENDPOINT_NAME}' deleted.")

# To also delete the registered model version from Unity Catalog:
# import mlflow
# mlflow_client = mlflow.tracking.MlflowClient(registry_uri="databricks-uc")
# mlflow_client.delete_registered_model(REGISTERED_MODEL_NAME)
# print(f"Registered model '{REGISTERED_MODEL_NAME}' deleted.")